In [1]:
!cp "/content/drive/MyDrive/Diplomamunka/SASRec/train_pids.npy" "/content/"

print("✅ Másolás kész! Az adatok a helyi lemezen vannak, az I/O sebesség maximális lesz.")

✅ Másolás kész! Az adatok a helyi lemezen vannak, az I/O sebesség maximális lesz.


In [2]:
!cp "/content/drive/MyDrive/Diplomamunka/SASRec/val_pids.npy" "/content/"

print("✅ Másolás kész! Az adatok a helyi lemezen vannak, az I/O sebesség maximális lesz.")

✅ Másolás kész! Az adatok a helyi lemezen vannak, az I/O sebesség maximális lesz.


In [4]:

!cp "/content/drive/MyDrive/Diplomamunka/SASRec/hybrid_embedding_matrix.npy" "/content/"

print("✅ Másolás kész! Az adatok a helyi lemezen vannak, az I/O sebesség maximális lesz.")

✅ Másolás kész! Az adatok a helyi lemezen vannak, az I/O sebesség maximális lesz.


In [10]:

!cp "/content/drive/MyDrive/Diplomamunka/SASRec/filtered_playlists_indices.pkl" "/content/"

print("✅ Másolás kész! Az adatok a helyi lemezen vannak, az I/O sebesség maximális lesz.")

✅ Másolás kész! Az adatok a helyi lemezen vannak, az I/O sebesség maximális lesz.


In [7]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

# --- KONFIGURÁCIÓ ---
HYBRID_MATRIX_PATH = "/content/hybrid_embedding_matrix.npy"
MAX_LEN = 50        # A lejátszási lista maximális hossza, amit a modell egyszerre lát
D_MODEL = 400       # A hibrid vektorok dimenziója
NUM_HEADS = 4       # Hány "szemmel" nézze a sorozatot az Attention
NUM_BLOCKS = 2      # Hány Transformer blokk legyen egymáson
DROPOUT_RATE = 0.2

# 1. Hibrid Embedding betöltése
print("📂 Hibrid mátrix betöltése...")
hybrid_weights = np.load(HYBRID_MATRIX_PATH).astype('float32')
VOCAB_SIZE = hybrid_weights.shape[0]

# --- MODELL KOMPONENSEK ---

def create_sasrec_model(vocab_size, d_model, max_len):
    # Bemenet: Dalok indexei a playlistben
    inputs = layers.Input(shape=(max_len,), name="input_ids")

    # 1. Embedding Réteg: Itt használjuk a Hibrid Mátrixot!
    # trainable=True, hogy a modell finomhangolhassa a CNN/W2V vektorokat
    embedding_layer = layers.Embedding(
        input_dim=vocab_size,
        output_dim=d_model,
        mask_zero=True,
        embeddings_initializer=tf.keras.initializers.Constant(hybrid_weights),
        trainable=True,
        name="hybrid_embedding"
    )
    x = embedding_layer(inputs)

    # 2. Positional Encoding: Hogy a modell tudja, mi a sorrend
    pos_indices = tf.range(max_len)[tf.newaxis, :]
    pos_embedding = layers.Embedding(max_len, d_model, name="position_embedding")(pos_indices)
    x += pos_embedding
    x = layers.Dropout(DROPOUT_RATE)(x)

    # 3. Transformer Blokkok (Self-Attention)
    for i in range(NUM_BLOCKS):
        # Multi-Head Attention
        attn_out = layers.MultiHeadAttention(
            num_heads=NUM_HEADS, key_dim=d_model, name=f"self_attn_{i}"
        )(x, x, use_causal_mask=True) # Causal mask: ne lásson a jövőbe!

        x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(DROPOUT_RATE)(attn_out))

        # Feed Forward Network
        ffn_out = layers.Dense(d_model, activation='relu')(x)
        ffn_out = layers.Dense(d_model)(ffn_out)

        x = layers.LayerNormalization(epsilon=1e-6)(x + layers.Dropout(DROPOUT_RATE)(ffn_out))

    # 4. Kimenet: Visszatérünk a vektortérbe
    # A cél, hogy a kimeneti vektor hasonlítson a következő dal embeddingjére
    outputs = layers.Dense(d_model, activation=None, name="output_projection")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    return model

# Modell példányosítása
model = create_sasrec_model(VOCAB_SIZE, D_MODEL, MAX_LEN)

# 5. Loss függvény: Koszinusz hasonlóság maximalizálása
# (Mivel vektorokat akarunk megjósolni, nem osztályokat)
def cosine_loss(y_true, y_pred):
    y_true_normalized = tf.math.l2_normalize(y_true, axis=-1)
    y_pred_normalized = tf.math.l2_normalize(y_pred, axis=-1)
    return 1.0 - tf.reduce_mean(tf.reduce_sum(y_true_normalized * y_pred_normalized, axis=-1))

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss=cosine_loss)

model.summary()

📂 Hibrid mátrix betöltése...


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_ids           │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hybrid_embedding    │ (None, 50, 400)   │ 148,973,2… │ input_ids[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 50, 400)   │          0 │ hybrid_embedding… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 50, 400)   │          0 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ self_attn_0         │ (None, 50, 400)   │  2,565,200 │ dropout[0][0],    │
│ (MultiHeadAttentio… │                   │            │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 50, 400)   │          0 │ self_attn_0[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 50, 400)   │          0 │ dropout[0][0],    │
│                     │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 50, 400)   │        800 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 50, 400)   │    160,400 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 50, 400)   │    160,400 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 50, 400)   │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 50, 400)   │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 50, 400)   │        800 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ self_attn_1         │ (None, 50, 400)   │  2,565,200 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 50, 400)   │          0 │ self_attn_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 50, 400)   │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 50, 400)   │        800 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 50, 400)   │    160,400 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 50, 400)   │    160,400 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 50, 400)   │          0 │ dense_3[0][0]   

 Total params: 154,908,800 (590.93 MB)

 Trainable params: 154,908,800 (590.93 MB)

 Non-trainable params: 0 (0.00 B)

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import load_model
import os
import gc

# --- 1. ÚTVONALAK ---
DRIVE_PATH = "/content/drive/MyDrive/Diplomamunka/SASRec"
HYBRID_MATRIX_PATH = os.path.join(DRIVE_PATH, "hybrid_embedding_matrix.npy")
TRAIN_DATA_PATH = os.path.join(DRIVE_PATH, "train_pids.npy")
VAL_DATA_PATH = os.path.join(DRIVE_PATH, "val_pids.npy")
SAVE_MODEL_PATH = os.path.join(DRIVE_PATH, "Models/best_sasrec_hybrid.weights.h5")

# --- 2. HIPERPARAMÉTEREK (Extra biztonságos) ---
BATCH_SIZE = 128  # 32-re csökkentve a memória miatt!
EPOCHS = 20
MAX_LEN = 50
D_MODEL = 400

# --- 3. ADATOK BETÖLTÉSE ---
print("📂 Adatok betöltése...")
hybrid_weights = np.load(HYBRID_MATRIX_PATH).astype('float32')
train_data = np.load(TRAIN_DATA_PATH, allow_pickle=True)
val_data = np.load(VAL_DATA_PATH, allow_pickle=True)
gc.collect()

# --- 4. GENERÁTOR ÉS LOSS (Változatlan) ---
class SASRecDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, playlists, embedding_matrix, batch_size=32, max_len=50, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.playlists = playlists
        self.embedding_matrix = embedding_matrix
        self.batch_size = batch_size
        self.max_len = max_len
        self.shuffle = shuffle
        self.indices = np.arange(len(self.playlists))
        if self.shuffle: np.random.shuffle(self.indices)

    def __len__(self): return int(np.ceil(len(self.playlists) / self.batch_size))

    def __getitem__(self, index):
        batch_idx = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        batch_input, batch_target = [], []
        for i in batch_idx:
            pl = self.playlists[i]
            if len(pl) < 2: continue
            seq = pl[:MAX_LEN+1]; input_seq = seq[:-1]; target_ids = seq[1:]
            pad_len = MAX_LEN - len(input_seq)
            batch_input.append([0] * pad_len + input_seq)
            target_vectors = [np.zeros(D_MODEL)] * pad_len
            for t_id in target_ids: target_vectors.append(self.embedding_matrix[t_id])
            batch_target.append(target_vectors)
        return np.array(batch_input), np.array(batch_target)

def masked_cosine_loss(y_true, y_pred):
    y_true_norm = tf.math.l2_normalize(y_true, axis=-1)
    y_pred_norm = tf.math.l2_normalize(y_pred, axis=-1)
    cosine_sim = tf.reduce_sum(y_true_norm * y_pred_norm, axis=-1)
    mask = tf.cast(tf.reduce_any(tf.not_equal(y_true, 0), axis=-1), tf.float32)
    return tf.reduce_sum((1.0 - cosine_sim) * mask) / (tf.reduce_sum(mask) + 1e-8)

# --- 5. MODELL LÉTREHOZÁSA (Ellenőrzéssel) ---
def create_model():
    inputs = layers.Input(shape=(MAX_LEN,))
    x = layers.Embedding(len(hybrid_weights), D_MODEL, mask_zero=True,
                         embeddings_initializer=tf.keras.initializers.Constant(hybrid_weights),
                         trainable=False)(inputs)
    x += layers.Embedding(MAX_LEN, D_MODEL)(tf.range(MAX_LEN)[tf.newaxis, :])
    for _ in range(2):
        attn = layers.MultiHeadAttention(4, D_MODEL)(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(attn))
        ffn = layers.Dense(D_MODEL * 4, activation='relu')(x)
        ffn = layers.Dense(D_MODEL)(ffn)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(ffn))
    return tf.keras.Model(inputs, layers.Dense(D_MODEL)(x))

# CSAK AKKOR TÖLTJÜK BE, HA LÉTEZIK ÉS NEM 0 BÁJTOS
if os.path.exists(SAVE_MODEL_PATH) and os.path.getsize(SAVE_MODEL_PATH) > 0:
    print("♻️ Érvényes modell betöltése...")
    model = load_model(SAVE_MODEL_PATH, custom_objects={'masked_cosine_loss': masked_cosine_loss})
else:
    print("🆕 Új modell indítása (a régi fájl sérült vagy nem létezik)...")
    model = create_model()
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss=masked_cosine_loss)

# --- 6. TANÍTÁS ---
train_gen = SASRecDataGenerator(train_data, hybrid_weights, batch_size=BATCH_SIZE)

sampled_val_data = val_data[::10]
val_gen = SASRecDataGenerator(sampled_val_data, hybrid_weights, batch_size=BATCH_SIZE)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(SAVE_MODEL_PATH, save_best_only=True, save_weights_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
]

model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks)

📂 Adatok betöltése...
🆕 Új modell indítása (a régi fájl sérült vagy nem létezik)...
Epoch 1/20
6199/6199 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step - loss: 0.2255
Epoch 1: val_loss improved from None to 0.21164, saving model to /content/drive/MyDrive/Diplomamunka/SASRec/Models/best_sasrec_hybrid.weights.h5

Epoch 1: finished saving model to /content/drive/MyDrive/Diplomamunka/SASRec/Models/best_sasrec_hybrid.weights.h5
6199/6199 ━━━━━━━━━━━━━━━━━━━━ 967s 154ms/step - loss: 0.2167 - val_loss: 0.2116
Epoch 2/20
6199/6199 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step - loss: 0.2129
Epoch 2: val_loss improved from 0.21164 to 0.21027, saving model to /content/drive/MyDrive/Diplomamunka/SASRec/Models/best_sasrec_hybrid.weights.h5

Epoch 2: finished saving model to /content/drive/MyDrive/Diplomamunka/SASRec/Models/best_sasrec_hybrid.weights.h5
6199/6199 ━━━━━━━━━━━━━━━━━━━━ 942s 152ms/step - loss: 0.2127 - val_loss: 0.2103
Epoch 3/20
6199/6199 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - loss: 0.2121
Epoch 3: val_loss 

KeyboardInterrupt: 

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, mixed_precision
import os
import gc

# --- 0. MIXED PRECISION BEKAPCSOLÁSA (Kaggle GPU-khoz kötelező!) ---
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print("⚡ Mixed Precision aktiválva: float16 használatban.")

# --- 1. ÚTVONALAK (Kaggle Specifikus!) ---
# BEMENET (Csak olvasható mappa)
INPUT_PATH = "/kaggle/input/models/bresgbor/sasrec/tensorflow2/default/1"
HYBRID_MATRIX_PATH = os.path.join(INPUT_PATH, "hybrid_embedding_matrix.npy")
TRAIN_DATA_PATH = os.path.join(INPUT_PATH, "train_pids.npy")
VAL_DATA_PATH = os.path.join(INPUT_PATH, "val_pids.npy")

# SÚLYOK KEZELÉSE
# Ha feltöltötted az eddigi mentett súlyokat az inputba, innen olvassuk be:
LOAD_MODEL_PATH = os.path.join(INPUT_PATH, "best_sasrec_hybrid.weights.h5") 
# Menteni viszont csak a working mappába tudunk!
SAVE_MODEL_PATH = "/kaggle/working/best_sasrec_hybrid.weights.h5"

# --- 2. HIPERPARAMÉTEREK ---
BATCH_SIZE = 64     # Kaggle-en a 30GB RAM miatt nyugodtan mehet 64-re is!
EPOCHS = 10         
MAX_LEN = 50
D_MODEL = 400
LEARNING_RATE = 1e-5 

# --- 3. ADATOK BETÖLTÉSE ---
print("📂 Adatok betöltése...")
hybrid_weights = np.load(HYBRID_MATRIX_PATH).astype('float32')
train_data = np.load(TRAIN_DATA_PATH, allow_pickle=True)
val_data = np.load(VAL_DATA_PATH, allow_pickle=True)
gc.collect()

# --- 4. GENERÁTOR ÉS LOSS ---
class SASRecDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, playlists, embedding_matrix, batch_size=32, max_len=50, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.playlists = playlists
        self.embedding_matrix = embedding_matrix
        self.batch_size = batch_size
        self.max_len = max_len
        self.shuffle = shuffle
        self.indices = np.arange(len(self.playlists))
        if self.shuffle: np.random.shuffle(self.indices)

    def __len__(self): return int(np.ceil(len(self.playlists) / self.batch_size))

    def __getitem__(self, index):
        batch_idx = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        batch_input, batch_target = [], []
        for i in batch_idx:
            pl = self.playlists[i]
            if len(pl) < 2: continue
            seq = pl[:MAX_LEN+1]; input_seq = seq[:-1]; target_ids = seq[1:]
            pad_len = MAX_LEN - len(input_seq)
            batch_input.append([0] * pad_len + input_seq)
            target_vectors = [np.zeros(D_MODEL)] * pad_len
            for t_id in target_ids: target_vectors.append(self.embedding_matrix[t_id])
            batch_target.append(target_vectors)
        return np.array(batch_input), np.array(batch_target)

def masked_cosine_loss(y_true, y_pred):
    # Stabilizáljuk a Loss-t float32-vel a Mixed Precision miatt
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    y_true_norm = tf.math.l2_normalize(y_true, axis=-1)
    y_pred_norm = tf.math.l2_normalize(y_pred, axis=-1)
    cosine_sim = tf.reduce_sum(y_true_norm * y_pred_norm, axis=-1)
    mask = tf.cast(tf.reduce_any(tf.not_equal(y_true, 0), axis=-1), tf.float32)
    return tf.reduce_sum((1.0 - cosine_sim) * mask) / (tf.reduce_sum(mask) + 1e-8)

# --- 5. MODELL FELÉPÍTÉSE ---
def create_model():
    inputs = layers.Input(shape=(MAX_LEN,))
    x = layers.Embedding(len(hybrid_weights), D_MODEL, mask_zero=True,
                         embeddings_initializer=tf.keras.initializers.Constant(hybrid_weights),
                         trainable=True, name="hybrid_embedding")(inputs)

    x += layers.Embedding(MAX_LEN, D_MODEL, name="pos_embedding")(tf.range(MAX_LEN)[tf.newaxis, :])

    for i in range(2):
        attn = layers.MultiHeadAttention(4, D_MODEL, name=f"attn_{i}")(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(attn))
        ffn = layers.Dense(D_MODEL * 4, activation='relu')(x)
        ffn = layers.Dense(D_MODEL)(ffn)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(ffn))

    # KRITIKUS: Az utolsó réteg maradjon float32!
    outputs = layers.Dense(D_MODEL, dtype='float32', name="final_projection")(x)
    return tf.keras.Model(inputs, outputs)

# --- 6. SÚLYOK BETÖLTÉSE ÉS FINE-TUNING INDÍTÁSA ---
model = create_model()

# Ha feltöltötted a korábbi súlyokat az Input mappába, onnan töltjük be:
if os.path.exists(LOAD_MODEL_PATH):
    print(f"♻️ Korábbi súlyok betöltése innen: {LOAD_MODEL_PATH}")
    model.load_weights(LOAD_MODEL_PATH)
    print("✅ Súlyok sikeresen ráhúzva a modellre.")
else:
    print("🆕 Nem található mentett súlyfájl, tiszta lappal indulunk.")

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
              loss=masked_cosine_loss)

# --- 7. TANÍTÁS ---
train_gen = SASRecDataGenerator(train_data, hybrid_weights, batch_size=BATCH_SIZE)
sampled_val_data = val_data[::50]
val_gen = SASRecDataGenerator(sampled_val_data, hybrid_weights, batch_size=BATCH_SIZE)

callbacks = [
    # MENTÉS A WORKING MAPPÁBA!
    tf.keras.callbacks.ModelCheckpoint(SAVE_MODEL_PATH, save_best_only=True, save_weights_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)
]

print(f"🚀 Fine-tuning indítása... Mentés helye: {SAVE_MODEL_PATH}")
model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks)

gc.collect()

2026-03-27 21:30:09.614614: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774647009.848545      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774647009.908021      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774647010.387784      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774647010.387839      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774647010.387845      55 computation_placer.cc:177] computation placer alr

⚡ Mixed Precision aktiválva: float16 használatban.
📂 Adatok betöltése...


I0000 00:00:1774647049.240236      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774647049.246869      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


♻️ Korábbi súlyok betöltése innen: /kaggle/input/models/bresgbor/sasrec/tensorflow2/default/1/best_sasrec_hybrid.weights.h5
✅ Súlyok sikeresen ráhúzva a modellre.
🚀 Fine-tuning indítása... Mentés helye: /kaggle/working/best_sasrec_hybrid.weights.h5
Epoch 1/10


/usr/local/lib/python3.12/dist-packages/tensorflow/python/framework/indexed_slices.py:446: UserWarning: Converting sparse IndexedSlices to a dense Tensor with 148973200 elements. This may consume a large amount of memory.
  warnings.warn(
I0000 00:00:1774647064.107062     128 service.cc:152] XLA service 0x7c100800a0b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774647064.107120     128 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1774647064.107128     128 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1774647065.719369     128 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1774647075.537303     128 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


12398/12398 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - loss: 0.2106
Epoch 1: val_loss improved from inf to 0.20848, saving model to /kaggle/working/best_sasrec_hybrid.weights.h5
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 860s 68ms/step - loss: 0.2106 - val_loss: 0.2085
Epoch 2/10
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 0.2082
Epoch 2: val_loss improved from 0.20848 to 0.20767, saving model to /kaggle/working/best_sasrec_hybrid.weights.h5
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 831s 67ms/step - loss: 0.2082 - val_loss: 0.2077
Epoch 3/10
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 0.2075
Epoch 3: val_loss improved from 0.20767 to 0.20717, saving model to /kaggle/working/best_sasrec_hybrid.weights.h5
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 826s 67ms/step - loss: 0.2075 - val_loss: 0.2072
Epoch 4/10
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - loss: 0.2067
Epoch 4: val_loss improved from 0.20717 to 0.20687, saving model to /kaggle/working/best_sasrec_hybrid.weights.h5
12398/12398 ━━━━━━━━━━━━━

1289

In [3]:
import os

target_path = "/content/drive/MyDrive/Diplomamunka/SASRec/Models/best_sasrec_hybrid.keras"

if os.path.exists(target_path):
    print(f"✅ A fájl létezik! Mérete: {os.path.getsize(target_path) / 1024 / 1024:.2f} MB")
else:
    print("❌ A fájl nem található ezen az úton!")
    # Listázzuk ki a szülőmappát, hogy lássuk, mi van benne:
    parent = os.path.dirname(target_path)
    if os.path.exists(parent):
        print(f"A mappában ({parent}) található fájlok: {os.listdir(parent)}")
    else:
        print(f"Még a mappa sem létezik: {parent}")

✅ A fájl létezik! Mérete: 0.00 MB


# SASRec továbbfejlesztése (Negatív Mintavételezés, BPR Loss, és Dot Product architektúra)

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, mixed_precision
import os
import gc

# --- 0. MIXED PRECISION BEKAPCSOLÁSA ---
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print("⚡ Mixed Precision aktiválva (BPR mód).")

# --- 1. ÚTVONALAK ---
INPUT_PATH = "/kaggle/input/models/bresgbor/sasrec/tensorflow2/default/1"
HYBRID_MATRIX_PATH = os.path.join(INPUT_PATH, "hybrid_embedding_matrix.npy")
TRAIN_DATA_PATH = os.path.join(INPUT_PATH, "train_pids.npy")
VAL_DATA_PATH = os.path.join(INPUT_PATH, "val_pids.npy")

# ÚJ NÉV, hogy ne írjuk felül a tegnapi 3.8%-os modellt!
SAVE_MODEL_PATH = "/kaggle/working/best_sasrec_bpr.weights.h5"

# --- 2. HIPERPARAMÉTEREK ---
BATCH_SIZE = 64
EPOCHS = 15
MAX_LEN = 50
D_MODEL = 400
LEARNING_RATE = 5e-5 # Kicsit bátrabb, mint a tegnapi, de még mindig biztonságos a fine-tuninghoz

# --- 3. ADATOK BETÖLTÉSE ---
print("📂 Adatok betöltése...")
hybrid_weights = np.load(HYBRID_MATRIX_PATH).astype('float32')
vocab_size = hybrid_weights.shape[0]
train_data = np.load(TRAIN_DATA_PATH, allow_pickle=True)
val_data = np.load(VAL_DATA_PATH, allow_pickle=True)
gc.collect()

# --- 4. ÚJ GENERÁTOR: NEGATÍV MINTAVÉTELEZÉSSEL ---
class SASRecBPRGenerator(tf.keras.utils.Sequence):
    def __init__(self, playlists, vocab_size, batch_size=32, max_len=50, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.playlists = playlists
        self.vocab_size = vocab_size
        self.batch_size = batch_size
        self.max_len = max_len
        self.shuffle = shuffle
        self.indices = np.arange(len(self.playlists))
        if self.shuffle: np.random.shuffle(self.indices)

    def __len__(self): return int(np.ceil(len(self.playlists) / self.batch_size))

    def __getitem__(self, index):
        batch_idx = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        
        batch_seq, batch_pos, batch_neg, batch_mask = [], [], [], []
        
        for i in batch_idx:
            pl = self.playlists[i]
            if len(pl) < 2: continue
                
            seq = pl[:self.max_len+1]
            input_seq = seq[:-1]   # A kontextus
            target_pos = seq[1:]   # A helyes következő dalok
            pad_len = self.max_len - len(input_seq)
            
            # Padding hozzáadása
            padded_input = [0] * pad_len + input_seq
            padded_pos = [0] * pad_len + target_pos
            
            # --- NEGATÍV MINTÁK SORSOLÁSA ---
            # Sorsolunk annyi rossz dalt, amennyi jó dal volt a szekvenciában
            padded_neg = [0] * pad_len
            for _ in range(len(target_pos)):
                neg_id = np.random.randint(1, self.vocab_size)
                while neg_id in pl: # Ne sorsoljunk olyat, ami amúgy benne van a listában!
                    neg_id = np.random.randint(1, self.vocab_size)
                padded_neg.append(neg_id)
                
            # Maszk készítése (1.0 ahol valós adat van, 0.0 a paddingnál)
            mask = [0.0] * pad_len + [1.0] * len(target_pos)
            
            batch_seq.append(padded_input)
            batch_pos.append(padded_pos)
            batch_neg.append(padded_neg)
            batch_mask.append(mask)
            
        # 3 külön bemenetünk van az új architektúrához!
        X = {
            "seq_inputs": np.array(batch_seq),
            "pos_inputs": np.array(batch_pos),
            "neg_inputs": np.array(batch_neg)
        }
        y = np.array(batch_mask) # A mask-ot küldjük be y_true-ként a loss funkciónak
        return X, y

# --- 5. BPR LOSS FUNKCIÓ (A Motorcsere) ---
def bpr_loss(y_true, y_pred):
    mask = tf.cast(y_true, tf.float32) # Ez a padding maszkunk
    
    # y_pred két oszlopból áll: a jó dal pontszáma (0) és a rossz dalé (1)
    pos_scores = y_pred[..., 0]
    neg_scores = y_pred[..., 1]
    
    # BPR Formula: -log(sigmoid(pos - neg)) -> Numerikusan stabilabb a softplus használata
    diff = pos_scores - neg_scores
    loss = tf.math.softplus(-diff)
    
    return tf.reduce_sum(loss * mask) / (tf.reduce_sum(mask) + 1e-8)

# --- 6. ÚJ ARCHITEKTÚRA (Dot Product a végén - Keras 3 javítással) ---
def create_bpr_model(vocab_size):
    # 3 Bemenet: A szekvencia, a jó válasz, és a rossz válasz
    seq_inputs = layers.Input(shape=(MAX_LEN,), name="seq_inputs")
    pos_inputs = layers.Input(shape=(MAX_LEN,), name="pos_inputs")
    neg_inputs = layers.Input(shape=(MAX_LEN,), name="neg_inputs")

    # KÖZÖS Hibrid Embedding (Ezt fogja mozgatni a tanítás!)
    item_embedding = layers.Embedding(
        vocab_size, D_MODEL, mask_zero=True,
        embeddings_initializer=tf.keras.initializers.Constant(hybrid_weights),
        trainable=True, name="hybrid_embedding"
    )
    pos_embedding = layers.Embedding(MAX_LEN, D_MODEL, name="pos_embedding")

    # 1. Szekvencia kódolása (A Transformer)
    x = item_embedding(seq_inputs)
    x += pos_embedding(tf.range(MAX_LEN)[tf.newaxis, :])
    
    for i in range(2):
        attn = layers.MultiHeadAttention(4, D_MODEL, name=f"attn_{i}")(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(attn))
        ffn = layers.Dense(D_MODEL * 4, activation='relu')(x)
        ffn = layers.Dense(D_MODEL)(ffn)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(ffn))
        
    seq_states = layers.Dense(D_MODEL, name="final_projection")(x)

    # 2. Vektorok lekérése a jó és rossz dalokhoz a közös térképből
    pos_emb = item_embedding(pos_inputs)
    neg_emb = item_embedding(neg_inputs)

    # 3. Mágneses pontozás (Lambda rétegbe csomagolva a TF hibák elkerülésére!)
    def dot_and_cast(tensors):
        seq, emb = tensors
        scores = tf.reduce_sum(seq * emb, axis=-1, keepdims=True)
        return tf.cast(scores, tf.float32)

    dot_layer = layers.Lambda(dot_and_cast, name="dot_scoring")
    
    pos_scores = dot_layer([seq_states, pos_emb])
    neg_scores = dot_layer([seq_states, neg_emb])

    # Összefűzzük a két pontszámot, hogy a BPR Loss megkapja mindkettőt
    outputs = layers.Concatenate(axis=-1, name="concat_scores")([pos_scores, neg_scores])

    train_model = tf.keras.Model(inputs=[seq_inputs, pos_inputs, neg_inputs], outputs=outputs)
    return train_model

# --- 7. MODELL FELÉPÍTÉSE ÉS TANÍTÁS ---
model = create_bpr_model(vocab_size)
# Tiszta lappal indulunk, szóval NINCS load_weights! 

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss=bpr_loss)

# Generátorok indítása
train_gen = SASRecBPRGenerator(train_data, vocab_size, batch_size=BATCH_SIZE)
val_gen = SASRecBPRGenerator(val_data[::50], vocab_size, batch_size=BATCH_SIZE)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(SAVE_MODEL_PATH, save_best_only=True, save_weights_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
]

print("🚀 BPR (Mágneses) Tanítás indítása Tiszta Lappal...")
model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks)

gc.collect()

⚡ Mixed Precision aktiválva (BPR mód).
📂 Adatok betöltése...
🚀 BPR (Mágneses) Tanítás indítása Tiszta Lappal...
Epoch 1/15
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - loss: 0.0632
Epoch 1: val_loss improved from inf to 0.02817, saving model to /kaggle/working/best_sasrec_bpr.weights.h5
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 795s 63ms/step - loss: 0.0632 - val_loss: 0.0282
Epoch 2/15
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 0.0266
Epoch 2: val_loss improved from 0.02817 to 0.02324, saving model to /kaggle/working/best_sasrec_bpr.weights.h5
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 769s 62ms/step - loss: 0.0266 - val_loss: 0.0232
Epoch 3/15
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 0.0220
Epoch 3: val_loss improved from 0.02324 to 0.01988, saving model to /kaggle/working/best_sasrec_bpr.weights.h5
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 769s 62ms/step - loss: 0.0220 - val_loss: 0.0199
Epoch 4/15
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 0.0195
Epoch 4: val_loss improve

1138

# Még tovább fejlesztett modell

## Hibrid téren tanítás

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, mixed_precision
import os
import gc

# ==========================================
# 0. KÖRNYEZET ÉS OPTIMALIZÁCIÓ
# ==========================================
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print("⚡ Mixed Precision aktiválva a gyorsabb tanításért.")

# ==========================================
# 1. ÚTVONALAK ÉS HIPERPARAMÉTEREK
# ==========================================
INPUT_PATH = "/kaggle/input/models/bresgbor/sasrec/tensorflow2/default/1"
HYBRID_MATRIX_PATH = os.path.join(INPUT_PATH, "hybrid_embedding_matrix.npy")
TRAIN_DATA_PATH = os.path.join(INPUT_PATH, "train_pids.npy")
VAL_DATA_PATH = os.path.join(INPUT_PATH, "val_pids.npy")

# ÚJ FÁJLNÉV a "Nagy Agyhoz"
SAVE_MODEL_PATH = "/kaggle/working/best_sasrec_bpr_large_hard.weights.h5"

# --- A TE BEÁLLÍTÁSAID ---
BATCH_SIZE = 64        # 64 biztonságos a nagyobb modellhez a Kaggle memóriájában
EPOCHS = 30            # Felemelve 30-ra
MAX_LEN = 50
D_MODEL = 400
LEARNING_RATE = 5e-5   # Kezdeti tanulási ráta

# ==========================================
# 2. ADATOK BETÖLTÉSE
# ==========================================
print("📂 Adatok betöltése...")
hybrid_weights = np.load(HYBRID_MATRIX_PATH).astype('float32')
vocab_size = hybrid_weights.shape[0]
train_data = np.load(TRAIN_DATA_PATH, allow_pickle=True)
val_data = np.load(VAL_DATA_PATH, allow_pickle=True)
gc.collect()

# ==========================================
# 3. HARD NEGATIVE MEDENCE (POOL) GENERÁLÁSA
# ==========================================
print("📊 Dalok népszerűségének kiszámítása...")
item_counts = np.zeros(vocab_size)
for pl in train_data:
    for item in pl:
        item_counts[item] += 1

# Word2Vec simítás (0.75 hatvány) a túl domináns slágerek ellen
smoothed_counts = np.power(item_counts, 0.75)
smoothed_counts[0] = 0.0 # A 0-s (padding) ID-t SOHA nem sorsoljuk!
p_neg = smoothed_counts / np.sum(smoothed_counts)

print("🎲 Hard Negative Pool generálása (10 millió elem)... Kérlek várj pár másodpercet.")
NEG_POOL_SIZE = 10_000_000
hard_neg_pool = np.random.choice(np.arange(vocab_size), size=NEG_POOL_SIZE, p=p_neg)
print("✅ Pool elkészült! Jöhet a generátor.")

# ==========================================
# 4. TURBÓZOTT GENERÁTOR (O(1) Negatív Szűréssel)
# ==========================================
class SASRecHardBPRGenerator(tf.keras.utils.Sequence):
    def __init__(self, playlists, vocab_size, neg_pool, batch_size=32, max_len=50, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.playlists = playlists
        self.vocab_size = vocab_size
        self.neg_pool = neg_pool
        self.pool_size = len(neg_pool)
        self.batch_size = batch_size
        self.max_len = max_len
        self.shuffle = shuffle
        self.indices = np.arange(len(self.playlists))
        if self.shuffle: np.random.shuffle(self.indices)

    def __len__(self): 
        return int(np.ceil(len(self.playlists) / self.batch_size))

    def __getitem__(self, index):
        batch_idx = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        batch_seq, batch_pos, batch_neg, batch_mask = [], [], [], []
        
        for i in batch_idx:
            pl = self.playlists[i]
            if len(pl) < 2: continue
                
            seq = pl[:self.max_len+1]
            input_seq = seq[:-1]
            target_pos = seq[1:]
            pad_len = self.max_len - len(input_seq)
            
            padded_input = [0] * pad_len + input_seq
            padded_pos = [0] * pad_len + target_pos
            
            # --- O(1) OPTIMALIZÁCIÓ A TE ÖTLETED ALAPJÁN ---
            pl_set = set(pl) # Egyszer alakítjuk át Hash Table-re a listát!
            padded_neg = [0] * pad_len
            
            for _ in range(len(target_pos)):
                # Villámgyors sorsolás a súlyozott medencéből
                neg_id = self.neg_pool[np.random.randint(0, self.pool_size)]
                
                # Villámgyors O(1) ellenőrzés
                while neg_id in pl_set: 
                    neg_id = self.neg_pool[np.random.randint(0, self.pool_size)]
                padded_neg.append(neg_id)
                
            mask = [0.0] * pad_len + [1.0] * len(target_pos)
            
            batch_seq.append(padded_input)
            batch_pos.append(padded_pos)
            batch_neg.append(padded_neg)
            batch_mask.append(mask)
            
        X = {
            "seq_inputs": np.array(batch_seq),
            "pos_inputs": np.array(batch_pos),
            "neg_inputs": np.array(batch_neg)
        }
        y = np.array(batch_mask)
        return X, y

# ==========================================
# 5. BPR LOSS
# ==========================================
def bpr_loss(y_true, y_pred):
    mask = tf.cast(y_true, tf.float32)
    pos_scores = y_pred[..., 0]
    neg_scores = y_pred[..., 1]
    diff = pos_scores - neg_scores
    loss = tf.math.softplus(-diff)
    return tf.reduce_sum(loss * mask) / (tf.reduce_sum(mask) + 1e-8)

# ==========================================
# 6. AZ "IZMOSÍTOTT" MODELL (3 Blokk, 8 Fej)
# ==========================================
def create_large_bpr_model(vocab_size):
    seq_inputs = layers.Input(shape=(MAX_LEN,), name="seq_inputs")
    pos_inputs = layers.Input(shape=(MAX_LEN,), name="pos_inputs")
    neg_inputs = layers.Input(shape=(MAX_LEN,), name="neg_inputs")

    item_embedding = layers.Embedding(
        vocab_size, D_MODEL, mask_zero=True,
        embeddings_initializer=tf.keras.initializers.Constant(hybrid_weights),
        trainable=True, name="hybrid_embedding"
    )
    pos_embedding = layers.Embedding(MAX_LEN, D_MODEL, name="pos_embedding")

    x = item_embedding(seq_inputs)
    x += pos_embedding(tf.range(MAX_LEN)[tf.newaxis, :])
    
    # --- 3 BLOKK ÉS 8 FEJ ---
    for i in range(3): 
        attn = layers.MultiHeadAttention(num_heads=8, key_dim=D_MODEL//8, name=f"attn_{i}")(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(attn))
        ffn = layers.Dense(D_MODEL * 4, activation='relu')(x)
        ffn = layers.Dense(D_MODEL)(ffn)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(ffn))
        
    seq_states = layers.Dense(D_MODEL, name="final_projection")(x)

    pos_emb = item_embedding(pos_inputs)
    neg_emb = item_embedding(neg_inputs)

    # Keras 3 kompatibilis Dot Product
    def dot_and_cast(tensors):
        seq, emb = tensors
        scores = tf.reduce_sum(seq * emb, axis=-1, keepdims=True)
        return tf.cast(scores, tf.float32)

    dot_layer = layers.Lambda(dot_and_cast, name="dot_scoring")
    
    pos_scores = dot_layer([seq_states, pos_emb])
    neg_scores = dot_layer([seq_states, neg_emb])

    outputs = layers.Concatenate(axis=-1, name="concat_scores")([pos_scores, neg_scores])
    return tf.keras.Model(inputs=[seq_inputs, pos_inputs, neg_inputs], outputs=outputs)

# ==========================================
# 7. ÖSSZESZERELÉS ÉS TANÍTÁS INDÍTÁSA
# ==========================================
print("🏗️ Nagy modell felépítése...")
model = create_large_bpr_model(vocab_size)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss=bpr_loss)

train_gen = SASRecHardBPRGenerator(train_data, vocab_size, hard_neg_pool, batch_size=BATCH_SIZE)
# A validációt is érdemes kicsit sűríteni (pl. ::20), hogy pontosabb képet kapjunk a csökkenésről
val_gen = SASRecHardBPRGenerator(val_data[::20], vocab_size, hard_neg_pool, batch_size=BATCH_SIZE)

# --- A TE PROFI CALLBACKJEID ---
callbacks = [
    # Menti a legjobb modellt
    tf.keras.callbacks.ModelCheckpoint(SAVE_MODEL_PATH, save_best_only=True, save_weights_only=True, verbose=1),
    
    # Ha 2 epochon át nem javul, felezi a rátát (LR = LR * 0.5)
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    
    # Ha 5 epochon át nem javul, megállítja az egészet, és visszatölti a legjobbat
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
]

print(f"🚀 HARD NEGATIVE + LARGE MODEL Tanítás indítása ({EPOCHS} Epoch, Batch: {BATCH_SIZE})...")
model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks)

gc.collect()

⚡ Mixed Precision aktiválva a gyorsabb tanításért.
📂 Adatok betöltése...
📊 Dalok népszerűségének kiszámítása...
🎲 Hard Negative Pool generálása (10 millió elem)... Kérlek várj pár másodpercet.
✅ Pool elkészült! Jöhet a generátor.
🏗️ Nagy modell felépítése...
🚀 HARD NEGATIVE + LARGE MODEL Tanítás indítása (30 Epoch, Batch: 64)...
Epoch 1/30
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - loss: 0.1637
Epoch 1: val_loss improved from inf to 0.08603, saving model to /kaggle/working/best_sasrec_bpr_large_hard.weights.h5
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 785s 62ms/step - loss: 0.1637 - val_loss: 0.0860 - learning_rate: 5.0000e-05
Epoch 2/30
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 0.0844
Epoch 2: val_loss improved from 0.08603 to 0.07311, saving model to /kaggle/working/best_sasrec_bpr_large_hard.weights.h5
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 756s 61ms/step - loss: 0.0844 - val_loss: 0.0731 - learning_rate: 5.0000e-05
Epoch 3/30
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 

81574

# W2V vektortéren tanítás

In [4]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, mixed_precision
import os
import gc

# ==========================================
# 0. KÖRNYEZET ÉS OPTIMALIZÁCIÓ
# ==========================================
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print("⚡ Mixed Precision aktiválva a gyorsabb tanításért.")

# ==========================================
# 1. ÚTVONALAK ÉS HIPERPARAMÉTEREK
# ==========================================
# ⚠️ FIGYELEM: Ellenőrizd, hogy a Kaggle-ön tényleg ezen az útvonalon van-e az új fájlod!
INPUT_PATH = "/kaggle/input/models/bresgbor/sasrec/tensorflow2/default/1" 
W2V_MATRIX_PATH = "/kaggle/input/models/bresgbor/sasrec-word2vec/tensorflow2/default/1/word2vec_matrix.npy"
TRAIN_DATA_PATH = os.path.join(INPUT_PATH, "train_pids.npy")
VAL_DATA_PATH = os.path.join(INPUT_PATH, "val_pids.npy")

# ÚJ FÁJLNÉV a "Tiszta Word2Vec" modellhez, hogy a Hibrid megmaradjon!
SAVE_MODEL_PATH = "/kaggle/working/best_sasrec_large_hard_w2v_only.weights.h5"

# --- A TE BEÁLLÍTÁSAID ---
BATCH_SIZE = 64
EPOCHS = 30
MAX_LEN = 50
LEARNING_RATE = 5e-5

# ==========================================
# 2. ADATOK BETÖLTÉSE (Csak W2V)
# ==========================================
print("📂 Word2Vec adatok betöltése...")
w2v_weights = np.load(W2V_MATRIX_PATH).astype('float32')
vocab_size = w2v_weights.shape[0]

# DINAMIKUS DIMENZIÓ (Automatikusan leolvassa a W2V fájlból, pl. 400)
D_MODEL = w2v_weights.shape[1] 
print(f"📏 Szótár: {vocab_size} dal | Dimenzió: {D_MODEL}")

train_data = np.load(TRAIN_DATA_PATH, allow_pickle=True)
val_data = np.load(VAL_DATA_PATH, allow_pickle=True)
gc.collect()

# ==========================================
# 3. HARD NEGATIVE MEDENCE (POOL) GENERÁLÁSA
# ==========================================
print("📊 Dalok népszerűségének kiszámítása...")
item_counts = np.zeros(vocab_size)
for pl in train_data:
    for item in pl:
        item_counts[item] += 1

# Word2Vec simítás (0.75 hatvány) a túl domináns slágerek ellen
smoothed_counts = np.power(item_counts, 0.75)
smoothed_counts[0] = 0.0 # A 0-s (padding) ID-t SOHA nem sorsoljuk!
p_neg = smoothed_counts / np.sum(smoothed_counts)

print("🎲 Hard Negative Pool generálása (10 millió elem)... Kérlek várj pár másodpercet.")
NEG_POOL_SIZE = 10_000_000
hard_neg_pool = np.random.choice(np.arange(vocab_size), size=NEG_POOL_SIZE, p=p_neg)
print("✅ Pool elkészült! Jöhet a generátor.")

# ==========================================
# 4. TURBÓZOTT GENERÁTOR (O(1) Negatív Szűréssel)
# ==========================================
class SASRecHardBPRGenerator(tf.keras.utils.Sequence):
    def __init__(self, playlists, vocab_size, neg_pool, batch_size=32, max_len=50, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.playlists = playlists
        self.vocab_size = vocab_size
        self.neg_pool = neg_pool
        self.pool_size = len(neg_pool)
        self.batch_size = batch_size
        self.max_len = max_len
        self.shuffle = shuffle
        self.indices = np.arange(len(self.playlists))
        if self.shuffle: np.random.shuffle(self.indices)

    def __len__(self): 
        return int(np.ceil(len(self.playlists) / self.batch_size))

    def __getitem__(self, index):
        batch_idx = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        batch_seq, batch_pos, batch_neg, batch_mask = [], [], [], []
        
        for i in batch_idx:
            pl = self.playlists[i]
            if len(pl) < 2: continue
                
            seq = pl[:self.max_len+1]
            input_seq = seq[:-1]
            target_pos = seq[1:]
            pad_len = self.max_len - len(input_seq)
            
            padded_input = [0] * pad_len + input_seq
            padded_pos = [0] * pad_len + target_pos
            
            # --- O(1) OPTIMALIZÁCIÓ ---
            pl_set = set(pl) 
            padded_neg = [0] * pad_len
            
            for _ in range(len(target_pos)):
                neg_id = self.neg_pool[np.random.randint(0, self.pool_size)]
                while neg_id in pl_set: 
                    neg_id = self.neg_pool[np.random.randint(0, self.pool_size)]
                padded_neg.append(neg_id)
                
            mask = [0.0] * pad_len + [1.0] * len(target_pos)
            
            batch_seq.append(padded_input)
            batch_pos.append(padded_pos)
            batch_neg.append(padded_neg)
            batch_mask.append(mask)
            
        X = {
            "seq_inputs": np.array(batch_seq),
            "pos_inputs": np.array(batch_pos),
            "neg_inputs": np.array(batch_neg)
        }
        y = np.array(batch_mask)
        return X, y

# ==========================================
# 5. BPR LOSS
# ==========================================
def bpr_loss(y_true, y_pred):
    mask = tf.cast(y_true, tf.float32)
    pos_scores = y_pred[..., 0]
    neg_scores = y_pred[..., 1]
    diff = pos_scores - neg_scores
    loss = tf.math.softplus(-diff)
    return tf.reduce_sum(loss * mask) / (tf.reduce_sum(mask) + 1e-8)

# ==========================================
# 6. AZ "IZMOSÍTOTT" MODELL (W2V Alapokon)
# ==========================================
def create_large_bpr_model(vocab_size):
    seq_inputs = layers.Input(shape=(MAX_LEN,), name="seq_inputs")
    pos_inputs = layers.Input(shape=(MAX_LEN,), name="pos_inputs")
    neg_inputs = layers.Input(shape=(MAX_LEN,), name="neg_inputs")

    item_embedding = layers.Embedding(
        vocab_size, D_MODEL, mask_zero=True,
        # ITT CSERÉLTÜK LE A MÁTRIXOT W2V-RE!
        embeddings_initializer=tf.keras.initializers.Constant(w2v_weights),
        trainable=True, name="w2v_embedding"
    )
    pos_embedding = layers.Embedding(MAX_LEN, D_MODEL, name="pos_embedding")

    x = item_embedding(seq_inputs)
    x += pos_embedding(tf.range(MAX_LEN)[tf.newaxis, :])
    
    # --- 3 BLOKK ÉS 8 FEJ ---
    for i in range(3): 
        attn = layers.MultiHeadAttention(num_heads=8, key_dim=D_MODEL//8, name=f"attn_{i}")(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(attn))
        ffn = layers.Dense(D_MODEL * 4, activation='relu')(x)
        ffn = layers.Dense(D_MODEL)(ffn)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(ffn))
        
    seq_states = layers.Dense(D_MODEL, name="final_projection")(x)

    pos_emb = item_embedding(pos_inputs)
    neg_emb = item_embedding(neg_inputs)

    def dot_and_cast(tensors):
        seq, emb = tensors
        scores = tf.reduce_sum(seq * emb, axis=-1, keepdims=True)
        return tf.cast(scores, tf.float32)

    dot_layer = layers.Lambda(dot_and_cast, name="dot_scoring")
    
    pos_scores = dot_layer([seq_states, pos_emb])
    neg_scores = dot_layer([seq_states, neg_emb])

    outputs = layers.Concatenate(axis=-1, name="concat_scores")([pos_scores, neg_scores])
    return tf.keras.Model(inputs=[seq_inputs, pos_inputs, neg_inputs], outputs=outputs)

# ==========================================
# 7. ÖSSZESZERELÉS ÉS TANÍTÁS INDÍTÁSA
# ==========================================
print("🏗️ W2V modell felépítése...")
model = create_large_bpr_model(vocab_size)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss=bpr_loss)

train_gen = SASRecHardBPRGenerator(train_data, vocab_size, hard_neg_pool, batch_size=BATCH_SIZE)
val_gen = SASRecHardBPRGenerator(val_data[::20], vocab_size, hard_neg_pool, batch_size=BATCH_SIZE)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(SAVE_MODEL_PATH, save_best_only=True, save_weights_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
]

print(f"🚀 TISZTA WORD2VEC Tanítás indítása ({EPOCHS} Epoch, Batch: {BATCH_SIZE})...")
model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks)

gc.collect()

⚡ Mixed Precision aktiválva a gyorsabb tanításért.
📂 Word2Vec adatok betöltése...
📏 Szótár: 372433 dal | Dimenzió: 400
📊 Dalok népszerűségének kiszámítása...
🎲 Hard Negative Pool generálása (10 millió elem)... Kérlek várj pár másodpercet.
✅ Pool elkészült! Jöhet a generátor.
🏗️ W2V modell felépítése...
🚀 TISZTA WORD2VEC Tanítás indítása (30 Epoch, Batch: 64)...
Epoch 1/30
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - loss: 0.1132
Epoch 1: val_loss improved from inf to 0.05849, saving model to /kaggle/working/best_sasrec_large_hard_w2v_only.weights.h5
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 761s 60ms/step - loss: 0.1132 - val_loss: 0.0585 - learning_rate: 5.0000e-05
Epoch 2/30
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 0.0602
Epoch 2: val_loss improved from 0.05849 to 0.05366, saving model to /kaggle/working/best_sasrec_large_hard_w2v_only.weights.h5
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 722s 58ms/step - loss: 0.0602 - val_loss: 0.0537 - learning_rate: 5.0000e-05
Epoch 3/30
12398/12398

114425

# W2V téren tanítás, de nem mozdulhatnak el a vektorok (trainable=false)

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, mixed_precision
import os
import gc

# ==========================================
# 0. KÖRNYEZET ÉS OPTIMALIZÁCIÓ
# ==========================================
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print("⚡ Mixed Precision aktiválva a gyorsabb tanításért.")

# ==========================================
# 1. ÚTVONALAK ÉS HIPERPARAMÉTEREK
# ==========================================
# ⚠️ FIGYELEM: Ellenőrizd, hogy a Kaggle-ön tényleg ezen az útvonalon van-e az új fájlod!
INPUT_PATH = "/kaggle/input/models/bresgbor/sasrec/tensorflow2/default/1" 
W2V_MATRIX_PATH = "/kaggle/input/models/bresgbor/sasrec-word2vec/tensorflow2/default/1/word2vec_matrix.npy"
TRAIN_DATA_PATH = os.path.join(INPUT_PATH, "train_pids.npy")
VAL_DATA_PATH = os.path.join(INPUT_PATH, "val_pids.npy")

# ÚJ FÁJLNÉV a "Tiszta Word2Vec" modellhez, hogy a Hibrid megmaradjon!
SAVE_MODEL_PATH = "/kaggle/working/best_sasrec_large_hard_w2v_frozen.weights.h5"

# --- A TE BEÁLLÍTÁSAID ---
BATCH_SIZE = 64
EPOCHS = 30
MAX_LEN = 50
LEARNING_RATE = 1e-4

# ==========================================
# 2. ADATOK BETÖLTÉSE (Csak W2V)
# ==========================================
print("📂 Word2Vec adatok betöltése...")
w2v_weights = np.load(W2V_MATRIX_PATH).astype('float32')
vocab_size = w2v_weights.shape[0]

# DINAMIKUS DIMENZIÓ (Automatikusan leolvassa a W2V fájlból, pl. 400)
D_MODEL = w2v_weights.shape[1] 
print(f"📏 Szótár: {vocab_size} dal | Dimenzió: {D_MODEL}")

train_data = np.load(TRAIN_DATA_PATH, allow_pickle=True)
val_data = np.load(VAL_DATA_PATH, allow_pickle=True)
gc.collect()

# ==========================================
# 3. HARD NEGATIVE MEDENCE (POOL) GENERÁLÁSA
# ==========================================
print("📊 Dalok népszerűségének kiszámítása...")
item_counts = np.zeros(vocab_size)
for pl in train_data:
    for item in pl:
        item_counts[item] += 1

# Word2Vec simítás (0.75 hatvány) a túl domináns slágerek ellen
smoothed_counts = np.power(item_counts, 0.75)
smoothed_counts[0] = 0.0 # A 0-s (padding) ID-t SOHA nem sorsoljuk!
p_neg = smoothed_counts / np.sum(smoothed_counts)

print("🎲 Hard Negative Pool generálása (10 millió elem)... Kérlek várj pár másodpercet.")
NEG_POOL_SIZE = 10_000_000
hard_neg_pool = np.random.choice(np.arange(vocab_size), size=NEG_POOL_SIZE, p=p_neg)
print("✅ Pool elkészült! Jöhet a generátor.")

# ==========================================
# 4. TURBÓZOTT GENERÁTOR (O(1) Negatív Szűréssel)
# ==========================================
class SASRecHardBPRGenerator(tf.keras.utils.Sequence):
    def __init__(self, playlists, vocab_size, neg_pool, batch_size=32, max_len=50, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.playlists = playlists
        self.vocab_size = vocab_size
        self.neg_pool = neg_pool
        self.pool_size = len(neg_pool)
        self.batch_size = batch_size
        self.max_len = max_len
        self.shuffle = shuffle
        self.indices = np.arange(len(self.playlists))
        if self.shuffle: np.random.shuffle(self.indices)

    def __len__(self): 
        return int(np.ceil(len(self.playlists) / self.batch_size))

    def __getitem__(self, index):
        batch_idx = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        batch_seq, batch_pos, batch_neg, batch_mask = [], [], [], []
        
        for i in batch_idx:
            pl = self.playlists[i]
            if len(pl) < 2: continue
                
            seq = pl[:self.max_len+1]
            input_seq = seq[:-1]
            target_pos = seq[1:]
            pad_len = self.max_len - len(input_seq)
            
            padded_input = [0] * pad_len + input_seq
            padded_pos = [0] * pad_len + target_pos
            
            # --- O(1) OPTIMALIZÁCIÓ ---
            pl_set = set(pl) 
            padded_neg = [0] * pad_len
            
            for _ in range(len(target_pos)):
                neg_id = self.neg_pool[np.random.randint(0, self.pool_size)]
                while neg_id in pl_set: 
                    neg_id = self.neg_pool[np.random.randint(0, self.pool_size)]
                padded_neg.append(neg_id)
                
            mask = [0.0] * pad_len + [1.0] * len(target_pos)
            
            batch_seq.append(padded_input)
            batch_pos.append(padded_pos)
            batch_neg.append(padded_neg)
            batch_mask.append(mask)
            
        X = {
            "seq_inputs": np.array(batch_seq),
            "pos_inputs": np.array(batch_pos),
            "neg_inputs": np.array(batch_neg)
        }
        y = np.array(batch_mask)
        return X, y

# ==========================================
# 5. BPR LOSS
# ==========================================
def bpr_loss(y_true, y_pred):
    mask = tf.cast(y_true, tf.float32)
    pos_scores = y_pred[..., 0]
    neg_scores = y_pred[..., 1]
    diff = pos_scores - neg_scores
    loss = tf.math.softplus(-diff)
    return tf.reduce_sum(loss * mask) / (tf.reduce_sum(mask) + 1e-8)

# ==========================================
# 6. AZ "IZMOSÍTOTT" MODELL (W2V Alapokon)
# ==========================================
def create_large_bpr_model(vocab_size):
    seq_inputs = layers.Input(shape=(MAX_LEN,), name="seq_inputs")
    pos_inputs = layers.Input(shape=(MAX_LEN,), name="pos_inputs")
    neg_inputs = layers.Input(shape=(MAX_LEN,), name="neg_inputs")

    item_embedding = layers.Embedding(
        vocab_size, D_MODEL, mask_zero=True,
        # ITT CSERÉLTÜK LE A MÁTRIXOT W2V-RE!
        embeddings_initializer=tf.keras.initializers.Constant(w2v_weights),
        trainable=False, name="w2v_embedding"
    )
    pos_embedding = layers.Embedding(MAX_LEN, D_MODEL, name="pos_embedding")

    x = item_embedding(seq_inputs)
    x += pos_embedding(tf.range(MAX_LEN)[tf.newaxis, :])
    
    # --- 3 BLOKK ÉS 8 FEJ ---
    for i in range(3): 
        attn = layers.MultiHeadAttention(num_heads=8, key_dim=D_MODEL//8, name=f"attn_{i}")(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(attn))
        ffn = layers.Dense(D_MODEL * 4, activation='relu')(x)
        ffn = layers.Dense(D_MODEL)(ffn)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(ffn))
        
    seq_states = layers.Dense(D_MODEL, name="final_projection")(x)

    pos_emb = item_embedding(pos_inputs)
    neg_emb = item_embedding(neg_inputs)

    def dot_and_cast(tensors):
        seq, emb = tensors
        scores = tf.reduce_sum(seq * emb, axis=-1, keepdims=True)
        return tf.cast(scores, tf.float32)

    dot_layer = layers.Lambda(dot_and_cast, name="dot_scoring")
    
    pos_scores = dot_layer([seq_states, pos_emb])
    neg_scores = dot_layer([seq_states, neg_emb])

    outputs = layers.Concatenate(axis=-1, name="concat_scores")([pos_scores, neg_scores])
    return tf.keras.Model(inputs=[seq_inputs, pos_inputs, neg_inputs], outputs=outputs)

# ==========================================
# 7. ÖSSZESZERELÉS ÉS TANÍTÁS INDÍTÁSA
# ==========================================
print("🏗️ W2V modell felépítése...")
model = create_large_bpr_model(vocab_size)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss=bpr_loss)

train_gen = SASRecHardBPRGenerator(train_data, vocab_size, hard_neg_pool, batch_size=BATCH_SIZE)
val_gen = SASRecHardBPRGenerator(val_data[::20], vocab_size, hard_neg_pool, batch_size=BATCH_SIZE)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(SAVE_MODEL_PATH, save_best_only=True, save_weights_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
]

print(f"🚀 TISZTA WORD2VEC Tanítás indítása ({EPOCHS} Epoch, Batch: {BATCH_SIZE})...")
model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks)

gc.collect()

2026-04-01 08:51:27.183305: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775033487.612295      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775033487.721921      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775033488.773672      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775033488.773713      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775033488.773715      55 computation_placer.cc:177] computation placer alr

⚡ Mixed Precision aktiválva a gyorsabb tanításért.
📂 Word2Vec adatok betöltése...
📏 Szótár: 372433 dal | Dimenzió: 400
📊 Dalok népszerűségének kiszámítása...
🎲 Hard Negative Pool generálása (10 millió elem)... Kérlek várj pár másodpercet.
✅ Pool elkészült! Jöhet a generátor.
🏗️ W2V modell felépítése...


I0000 00:00:1775033552.471429      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775033552.477463      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


🚀 TISZTA WORD2VEC Tanítás indítása (30 Epoch, Batch: 64)...
Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'dot_scoring' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
I0000 00:00:1775033563.720309     153 service.cc:152] XLA service 0x7a1f94001f20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775033563.720346     153 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1775033563.720353     153 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1775033565.585702     153 cuda_dnn.cc:529] Loaded cuDNN version 91002


    3/12398 ━━━━━━━━━━━━━━━━━━━━ 8:08 39ms/step - loss: 1.9336   

I0000 00:00:1775033574.825871     153 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


12398/12398 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 0.0989
Epoch 1: val_loss improved from inf to 0.05754, saving model to /kaggle/working/best_sasrec_large_hard_w2v_frozen.weights.h5
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 330s 25ms/step - loss: 0.0989 - val_loss: 0.0575 - learning_rate: 1.0000e-04
Epoch 2/30
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0584
Epoch 2: val_loss improved from 0.05754 to 0.05415, saving model to /kaggle/working/best_sasrec_large_hard_w2v_frozen.weights.h5
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 302s 24ms/step - loss: 0.0584 - val_loss: 0.0541 - learning_rate: 1.0000e-04
Epoch 3/30
12396/12398 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0551
Epoch 3: val_loss did not improve from 0.05415
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 297s 24ms/step - loss: 0.0551 - val_loss: 0.0544 - learning_rate: 1.0000e-04
Epoch 4/30
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.0534
Epoch 4: val_loss improved from 0.05415 to 0.05201, saving model to /kaggle/working/best_sas

4525

# Training from scratch baseline modell tanítás

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, mixed_precision
import os
import gc

# ==========================================
# 0. KÖRNYEZET ÉS OPTIMALIZÁCIÓ
# ==========================================
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print("⚡ Mixed Precision aktiválva a gyorsabb tanításért.")

# ==========================================
# 1. ÚTVONALAK ÉS HIPERPARAMÉTEREK
# ==========================================
# ⚠️ FIGYELEM: Ellenőrizd, hogy a Kaggle-ön tényleg ezen az útvonalon van-e az új fájlod!
INPUT_PATH = "/kaggle/input/models/bresgbor/sasrec/tensorflow2/default/1" 
W2V_MATRIX_PATH = "/kaggle/input/models/bresgbor/sasrec-word2vec/tensorflow2/default/1/word2vec_matrix.npy"
TRAIN_DATA_PATH = os.path.join(INPUT_PATH, "train_pids.npy")
VAL_DATA_PATH = os.path.join(INPUT_PATH, "val_pids.npy")

# ÚJ FÁJLNÉV a "Tiszta Word2Vec" modellhez, hogy a Hibrid megmaradjon!
SAVE_MODEL_PATH = "/kaggle/working/best_sasrec_baseline_scratch.weights.h5"

# --- A TE BEÁLLÍTÁSAID ---
BATCH_SIZE = 64
EPOCHS = 30
MAX_LEN = 50
LEARNING_RATE = 1e-3

# ==========================================
# 2. ADATOK BETÖLTÉSE (Csak W2V)
# ==========================================
print("📂 Word2Vec adatok betöltése...")
w2v_weights = np.load(W2V_MATRIX_PATH).astype('float32')
vocab_size = w2v_weights.shape[0]
del w2v_weights

# DINAMIKUS DIMENZIÓ (Automatikusan leolvassa a W2V fájlból, pl. 400)
D_MODEL = 256 
print(f"📏 Szótár: {vocab_size} dal | Dimenzió: {D_MODEL}")

train_data = np.load(TRAIN_DATA_PATH, allow_pickle=True)
val_data = np.load(VAL_DATA_PATH, allow_pickle=True)
gc.collect()

# ==========================================
# 3. HARD NEGATIVE MEDENCE (POOL) GENERÁLÁSA
# ==========================================
print("📊 Dalok népszerűségének kiszámítása...")
item_counts = np.zeros(vocab_size)
for pl in train_data:
    for item in pl:
        item_counts[item] += 1

# Word2Vec simítás (0.75 hatvány) a túl domináns slágerek ellen
smoothed_counts = np.power(item_counts, 0.75)
smoothed_counts[0] = 0.0 # A 0-s (padding) ID-t SOHA nem sorsoljuk!
p_neg = smoothed_counts / np.sum(smoothed_counts)

print("🎲 Hard Negative Pool generálása (10 millió elem)... Kérlek várj pár másodpercet.")
NEG_POOL_SIZE = 10_000_000
hard_neg_pool = np.random.choice(np.arange(vocab_size), size=NEG_POOL_SIZE, p=p_neg)
print("✅ Pool elkészült! Jöhet a generátor.")

# ==========================================
# 4. TURBÓZOTT GENERÁTOR (O(1) Negatív Szűréssel)
# ==========================================
class SASRecHardBPRGenerator(tf.keras.utils.Sequence):
    def __init__(self, playlists, vocab_size, neg_pool, batch_size=32, max_len=50, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.playlists = playlists
        self.vocab_size = vocab_size
        self.neg_pool = neg_pool
        self.pool_size = len(neg_pool)
        self.batch_size = batch_size
        self.max_len = max_len
        self.shuffle = shuffle
        self.indices = np.arange(len(self.playlists))
        if self.shuffle: np.random.shuffle(self.indices)

    def __len__(self): 
        return int(np.ceil(len(self.playlists) / self.batch_size))

    def __getitem__(self, index):
        batch_idx = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        batch_seq, batch_pos, batch_neg, batch_mask = [], [], [], []
        
        for i in batch_idx:
            pl = self.playlists[i]
            if len(pl) < 2: continue
                
            seq = pl[:self.max_len+1]
            input_seq = seq[:-1]
            target_pos = seq[1:]
            pad_len = self.max_len - len(input_seq)
            
            padded_input = [0] * pad_len + input_seq
            padded_pos = [0] * pad_len + target_pos
            
            # --- O(1) OPTIMALIZÁCIÓ ---
            pl_set = set(pl) 
            padded_neg = [0] * pad_len
            
            for _ in range(len(target_pos)):
                neg_id = self.neg_pool[np.random.randint(0, self.pool_size)]
                while neg_id in pl_set: 
                    neg_id = self.neg_pool[np.random.randint(0, self.pool_size)]
                padded_neg.append(neg_id)
                
            mask = [0.0] * pad_len + [1.0] * len(target_pos)
            
            batch_seq.append(padded_input)
            batch_pos.append(padded_pos)
            batch_neg.append(padded_neg)
            batch_mask.append(mask)
            
        X = {
            "seq_inputs": np.array(batch_seq),
            "pos_inputs": np.array(batch_pos),
            "neg_inputs": np.array(batch_neg)
        }
        y = np.array(batch_mask)
        return X, y

# ==========================================
# 5. BPR LOSS
# ==========================================
def bpr_loss(y_true, y_pred):
    mask = tf.cast(y_true, tf.float32)
    pos_scores = y_pred[..., 0]
    neg_scores = y_pred[..., 1]
    diff = pos_scores - neg_scores
    loss = tf.math.softplus(-diff)
    return tf.reduce_sum(loss * mask) / (tf.reduce_sum(mask) + 1e-8)

# ==========================================
# 6. AZ "IZMOSÍTOTT" MODELL (W2V Alapokon)
# ==========================================
def create_large_bpr_model(vocab_size):
    seq_inputs = layers.Input(shape=(MAX_LEN,), name="seq_inputs")
    pos_inputs = layers.Input(shape=(MAX_LEN,), name="pos_inputs")
    neg_inputs = layers.Input(shape=(MAX_LEN,), name="neg_inputs")

    item_embedding = layers.Embedding(
        vocab_size, D_MODEL, mask_zero=True,
        trainable=True, name="item_embedding"
    )
    pos_embedding = layers.Embedding(MAX_LEN, D_MODEL, name="pos_embedding")

    x = item_embedding(seq_inputs)
    x += pos_embedding(tf.range(MAX_LEN)[tf.newaxis, :])
    
    # --- 3 BLOKK ÉS 8 FEJ ---
    for i in range(3): 
        attn = layers.MultiHeadAttention(num_heads=8, key_dim=D_MODEL//8, name=f"attn_{i}")(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(attn))
        ffn = layers.Dense(D_MODEL * 4, activation='relu')(x)
        ffn = layers.Dense(D_MODEL)(ffn)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(ffn))
        
    seq_states = layers.Dense(D_MODEL, name="final_projection")(x)

    pos_emb = item_embedding(pos_inputs)
    neg_emb = item_embedding(neg_inputs)

    def dot_and_cast(tensors):
        seq, emb = tensors
        scores = tf.reduce_sum(seq * emb, axis=-1, keepdims=True)
        return tf.cast(scores, tf.float32)

    dot_layer = layers.Lambda(dot_and_cast, name="dot_scoring")
    
    pos_scores = dot_layer([seq_states, pos_emb])
    neg_scores = dot_layer([seq_states, neg_emb])

    outputs = layers.Concatenate(axis=-1, name="concat_scores")([pos_scores, neg_scores])
    return tf.keras.Model(inputs=[seq_inputs, pos_inputs, neg_inputs], outputs=outputs)

# ==========================================
# 7. ÖSSZESZERELÉS ÉS TANÍTÁS INDÍTÁSA
# ==========================================
print("🏗️ W2V modell felépítése...")
model = create_large_bpr_model(vocab_size)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss=bpr_loss)

train_gen = SASRecHardBPRGenerator(train_data, vocab_size, hard_neg_pool, batch_size=BATCH_SIZE)
val_gen = SASRecHardBPRGenerator(val_data[::20], vocab_size, hard_neg_pool, batch_size=BATCH_SIZE)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(SAVE_MODEL_PATH, save_best_only=True, save_weights_only=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
]

print(f"🚀 TISZTA WORD2VEC Tanítás indítása ({EPOCHS} Epoch, Batch: {BATCH_SIZE})...")
model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS, callbacks=callbacks)

gc.collect()

2026-04-01 11:59:58.330583: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775044798.517761      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775044798.568586      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775044798.987170      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775044798.987210      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775044798.987213      55 computation_placer.cc:177] computation placer alr

⚡ Mixed Precision aktiválva a gyorsabb tanításért.
📂 Word2Vec adatok betöltése...
📏 Szótár: 372433 dal | Dimenzió: 256
📊 Dalok népszerűségének kiszámítása...
🎲 Hard Negative Pool generálása (10 millió elem)... Kérlek várj pár másodpercet.
✅ Pool elkészült! Jöhet a generátor.
🏗️ W2V modell felépítése...


I0000 00:00:1775044856.793668      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775044856.800957      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


🚀 TISZTA WORD2VEC Tanítás indítása (30 Epoch, Batch: 64)...
Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'dot_scoring' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
I0000 00:00:1775044867.736188     216 service.cc:152] XLA service 0x792dbc00b000 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775044867.736229     216 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1775044867.736233     216 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1775044869.633636     216 cuda_dnn.cc:529] Loaded cuDNN version 91002


    3/12398 ━━━━━━━━━━━━━━━━━━━━ 9:11 45ms/step - loss: 0.7469   

I0000 00:00:1775044878.073310     216 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


12398/12398 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.2824
Epoch 1: val_loss improved from inf to 0.11379, saving model to /kaggle/working/best_sasrec_baseline_scratch.weights.h5
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 450s 35ms/step - loss: 0.2824 - val_loss: 0.1138 - learning_rate: 0.0010
Epoch 2/30
12397/12398 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 0.1037
Epoch 2: val_loss improved from 0.11379 to 0.09861, saving model to /kaggle/working/best_sasrec_baseline_scratch.weights.h5
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 422s 34ms/step - loss: 0.1037 - val_loss: 0.0986 - learning_rate: 0.0010
Epoch 3/30
12397/12398 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - loss: 0.0881
Epoch 3: val_loss improved from 0.09861 to 0.09122, saving model to /kaggle/working/best_sasrec_baseline_scratch.weights.h5
12398/12398 ━━━━━━━━━━━━━━━━━━━━ 421s 34ms/step - loss: 0.0881 - val_loss: 0.0912 - learning_rate: 0.0010
Epoch 4/30
12397/12398 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.0806
Epoch 4: val_loss improved from 0.091

4723